In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import os

In [13]:
# 하이퍼파라미터 설정
BATCH_SIZE=32
EPOCHS=3
LR =0.01
NUM_CLASSES =2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [14]:
# GPU 사용 가능 여부 확인
print(f"GPU 사용 가능 여부: {torch.cuda.is_available()}")

# 사용 가능한 GPU 개수 및 이름 확인
if torch.cuda.is_available():
    print(f"GPU 개수: {torch.cuda.device_count()}")
    print(f"현재 GPU 이름: {torch.cuda.get_device_name(0)}")


GPU 사용 가능 여부: True
GPU 개수: 4
현재 GPU 이름: NVIDIA TITAN Xp


In [15]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # 사용할 GPU 번호 입력
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [16]:
import glob 
import os 

# 경로 지정 
root='./'
image_folder_path = os.path.join(root, 'product_images') 

# images 폴더가 없으면 생성(이미 존재하면 건너뜀)
if not os.path.exists(image_folder_path):
    os.makedirs(image_folder_path)

import kaggle
kaggle.api.dataset_download_files('ravirajsinh45/real-life-industrial-dataset-of-casting-product', path=image_folder_path, unzip=True)

print(image_folder_path)
print(glob.glob(image_folder_path+'/*')) 

Dataset URL: https://www.kaggle.com/datasets/ravirajsinh45/real-life-industrial-dataset-of-casting-product
./product_images
['./product_images/casting_data', './product_images/casting_512x512']


In [17]:
TRAIN_DIR = os.path.join(image_folder_path, "casting_data", "casting_data","train")
TEST_DIR = os.path.join(image_folder_path, "casting_data", "casting_data", "test")

print("Train 경로:", TRAIN_DIR)
print("TEST 경로:", TEST_DIR)

Train 경로: ./product_images/casting_data/casting_data/train
TEST 경로: ./product_images/casting_data/casting_data/test


In [18]:
# VGGNet 19-layer (VGG-19) 정의 
class VGG19(nn.Module):
    def __init__(self, num_classes=1000):
        super(VGG19, self).__init__()

        # Convolution + ReLU layers (3x3 convolution, padding=1)
        # nn.Sequential는 "레이어를 순서대로 담는 상자"이다.

        self.features = nn.Sequential(
            # Block 1 : 2 conv layers + maxpool
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 2: 2 conv layers + maxpool
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 3: 4 conv layers + maxpool
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 4 : 4 conv layers + maxpool
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 5 : 4 conv layers + maxpool
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # Fully Connected layers
        self.classifier = nn.Sequential(
            nn.Linear(512*7*7, 4096), # 입력 크기 : 마지막 feature map 7x7
            nn.ReLU(inplace=True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(),
            nn.Linear(4096, num_classes)
        )

    def forward(self,x):
        x=self.features(x) # Convolution & Pooling
        x=x.view(x.size(0),-1) # Flatten
        x=self.classifier(x) # Fully Connected Network
        return x
    
model = VGG19(num_classes=2).to(DEVICE)
print(f"Using device : {DEVICE}")
# print(model)

Using device : cuda


In [19]:
# 데이터 전처리 & DataLoader
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(), # 좌우 반전(데이터 증강)
    transforms.RandomRotation(10), # 10도 회전(데이터 증강)
    transforms.ToTensor(), 
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # ImageNet 기준 정규화
])

test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size = BATCH_SIZE, shuffle=False)

print(f"클래스 : {train_dataset.classes}")
print(f"학습 데이터 수: {len(train_dataset)}")
print(f"테스트 데이터 수: {len(test_dataset)}")

클래스 : ['def_front', 'ok_front']
학습 데이터 수: 6633
테스트 데이터 수: 715


In [20]:
# loss 함수 & optimizer 설정 
criterion = nn.CrossEntropyLoss() # softmax 내장 
optimizer = torch.optim.SGD(model.parameters(), lr=LR)

# train 
def train(model, loader, criterion, optimizer):
    model.train() # 학습 모드
    total_loss, correct, total = 0,0,0

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad() # gradient 초기화 
        outputs=model(images)
        loss=criterion(outputs, labels)
        loss.backward() 
        optimizer.step() # 가중치 업데이트 

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss/len(loader)
    accuracy = 100. *correct/total
    return avg_loss, accuracy

In [21]:
# test
def test(model, loader, criterion):
    model.eval() # 평가모드
    total_loss, correct, total = 0,0,0

    with torch.no_grad(): # gradient 계산 비활성화
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

        avg_loss = total_loss/len(loader)
        accuracy = 100. *correct/total
        return avg_loss, accuracy

In [22]:
# 학습 실행 
train_losses, train_accs = [], []
test_losses, test_accs = [], []

for epoch in range(1, EPOCHS +1):
    train_loss, train_acc = train(model, train_loader, criterion, optimizer)
    test_loss, test_acc = test(model, test_loader, criterion)

    train_losses.append(train_loss)
    train_accs.append(train_acc)
    test_losses.append(test_loss)
    test_accs.append(test_acc)

    print(f"Epoch [{epoch:02d}/{EPOCHS}] "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
          f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%")

Epoch [01/3] Train Loss: 0.6873 | Train Acc: 56.61% | Test Loss: 0.6750 | Test Acc: 63.36%
Epoch [02/3] Train Loss: 0.6847 | Train Acc: 56.66% | Test Loss: 0.6721 | Test Acc: 63.36%
Epoch [03/3] Train Loss: 0.6842 | Train Acc: 56.66% | Test Loss: 0.6716 | Test Acc: 63.36%
